<img src="logo.png" alt="Vegeta" width="240">

# Exotic propellers — six and twelve blades, tractor and pusher (25c)

Notebook 25's blade, six and twelve of it on one hub (an open "propfan"), on the same pod and pylon, as a tractor and as a
pusher. More blades of the same blade mean more solidity: the propeller absorbs the same power at a much lower rpm — a lower
tip speed — and every blade works at a lower lift; the cost is more profile drag and more mass.

**This notebook computes only the six- and twelve-blade propellers.** The two- and three-blade ones it is compared with come
from notebook 25's export (`designs/data/air_propeller_results.json`, its section 13) — the same installation model, the same
definitions; if the file is not there, the comparison is left out. The design inputs (blade, section, pod, aircraft, flight
points) are read from the same file so both notebooks describe the same propeller; without it they are notebook 25's defaults.

```
1  the design (from notebook 25's export), the propellers in CAD with the pod
2  the air the blades meet (the installation wake) and the installed model
3  full-power thrust, level-flight efficiency, rpm and blade passing — 6 and 12 blades against 2 and 3
4  the exotic-propeller maps for the flight models (MERLIN)
```

Blade-element theory with isolated-aerofoil data is generous to a twelve-blade hub (no cascade correction at that solidity):
read the twelve-blade numbers as an upper bound.

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, shutil, sys, dataclasses
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyvista as pv
from IPython.display import display
from vegeta import dedalus, boreas
from vegeta.boreas import wake
from vegeta.dedalus import viz as dviz
from vegeta.aeromant import viz as aviz
from vegeta.aeromant.stl import read_stl
from vegeta.dedalus.examples import Propeller as PropellerCAD
sys.path.insert(0, "designs")
import air_propeller as ap

RUNS = Path("_runs/exotic_propellers"); shutil.rmtree(RUNS, ignore_errors=True); (RUNS / "cad").mkdir(parents=True)
RHO, NU, G = 1.225, 1.5e-5, 9.81
RESULTS_JSON = Path("designs/data/air_propeller_results.json")
PJ = json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.exists() else None
COMPARE = PJ is not None and all(c.get("level_flight") for c in PJ["configs"].values())
print("two and three blades to compare with:", "notebook 25 export" if COMPARE else "none (run notebook 25 to export them)")

## 1. The design and the propellers

In [ ]:
BLADE = dict(diameter_mm=254.0, pitch_mm=152.4, chord_root_mm=18.0, chord_max_mm=26.0, chord_tip_mm=10.0, thickness=0.10, camber=0.04)
HUB = dict(hub_diameter=20.0, hub_height=10.0, bore=5.0)
SECTION = boreas.Airfoil(name="thin cambered section", cl_alpha=2 * math.pi * 0.9, alpha0_deg=-3.0, cl_max=1.1, cd0=0.018, k=0.04,
                         source="assumed for a 10-inch blade at Re ~ 1e5 (notebook 25)")
POD = dict(pod_diameter=46.0, pod_length=240.0, pylon_chord=90.0, pylon_thickness=0.12, pylon_height=230.0, pylon_gap=64.0)
PYLON_CD = 0.012
AIRCRAFT = dict(mass_kg=3.0, wing_area_m2=0.50, cd0=0.035, aspect_ratio=8.0, oswald=0.8)
POINTS = {"cruise": (20.0, 0.0), "climb": (14.0, 3.0), "take-off": (0.0, 0.0)}
SPEEDS = (10.0, 15.0, 20.0, 25.0, 30.0)
P_REF_SHAFT, TIP_MACH_LIMIT = 1900.0 * 0.85, 0.75
if PJ is not None:                                   # the same propeller and aircraft as notebook 25
    BLADE, HUB, POD, PYLON_CD, AIRCRAFT = PJ["blade"], PJ["hub"], PJ["pod"], PJ["pylon_cd"], PJ["aircraft"]
    SECTION = boreas.Airfoil(**PJ["section"])
    POINTS = {k: tuple(v) for k, v in PJ["points"].items()}
    SPEEDS = tuple(PJ["speeds"])
    if PJ.get("full_power_reference"):
        P_REF_SHAFT, TIP_MACH_LIMIT = PJ["full_power_reference"]["shaft_W"], PJ["full_power_reference"]["tip_mach"]
MANY = (6, 12)
LAYOUTS = ("tractor", "pusher")
BLADE_COLOR = {2: "#1f77b4", 3: "#9467bd", 6: "#2ca02c", 12: "#ff7f0e"}     # one colour per blade count (red is the ducted fan's)
LAYOUT_STYLE = {"pusher": dict(ls="-", lw=2.4), "tractor": dict(ls="--", lw=1.3)}

def required_thrust(V, climb_rate=0.0):
    a = AIRCRAFT
    W = a["mass_kg"] * G
    if V <= 0:
        return 0.75 * W
    q = 0.5 * RHO * V**2
    cl = W / (q * a["wing_area_m2"])
    k = 1 / (math.pi * a["aspect_ratio"] * a["oswald"])
    return q * a["wing_area_m2"] * (a["cd0"] + k * cl**2) + W * climb_rate / V

def prop_of(B):
    return boreas.Propeller.from_pitch(f"10x6, {B} blades", BLADE["diameter_mm"] / 1000, BLADE["pitch_mm"] / 1000, blades=B,
                                       chord_root_m=BLADE["chord_root_mm"] / 1000, chord_max_m=BLADE["chord_max_mm"] / 1000,
                                       chord_tip_m=BLADE["chord_tip_mm"] / 1000, hub_radius_m=HUB["hub_diameter"] / 2000)

def cad_kw(B):
    return dict(diameter=BLADE["diameter_mm"], pitch=BLADE["pitch_mm"], blades=B, chord_root=BLADE["chord_root_mm"],
                chord_max=BLADE["chord_max_mm"], chord_tip=BLADE["chord_tip_mm"], thickness=BLADE["thickness"], camber=BLADE["camber"],
                stations=8, **HUB)

PROPS = {B: prop_of(B) for B in MANY}
PROP_STL, PROP_MESH = {}, {}
for B in MANY:
    cad = PropellerCAD().generate(**cad_kw(B))
    px = dedalus.Geometry.from_cadquery(cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name=f"prop{B}_axis_x")
    PROP_STL[B] = px.export_stl(RUNS / "cad" / f"prop{B}_axis_x.stl", tolerance=0.03)
    PROP_MESH[B] = dviz.to_pyvista(px, 0.05).scale(0.001, inplace=False)
EXT = [read_stl(PROP_STL[B]).bbox for B in MANY]
X_AFT, X_FWD = max(b[1][0] for b in EXT), min(b[0][0] for b in EXT)
POD_GAP = {"tractor": X_AFT - HUB["hub_height"] / 2 + 2.0, "pusher": -X_FWD - HUB["hub_height"] / 2 + 2.0}
if PJ is not None:                                   # the same pod as notebook 25 (the same blade: the same extent)
    POD_GAP = {k: max(v, PJ["pod_gap_mm"][k]) for k, v in POD_GAP.items()}

def pod_p(lay):
    return dict(POD, layout=lay, hub_height=HUB["hub_height"], hub_diameter=HUB["hub_diameter"], gap=POD_GAP[lay])

pl = pv.Plotter(shape=(1, 2), window_size=(1200, 440))
for i, B in enumerate(MANY):
    pl.subplot(0, i)
    pl.add_mesh(dviz.to_pyvista(ap.PropPod().generate(**pod_p("tractor")), 0.2).scale(0.001, inplace=False), color="#c9c9c9", smooth_shading=True)
    pl.add_mesh(PROP_MESH[B], color=BLADE_COLOR[B], smooth_shading=True)
    pl.add_text(f"{B} blades, tractor", font_size=10)
    pl.camera_position = [(-0.35, -0.55, 0.30), (0.0, 0.05, 0.0), (0, 1, 0)]
pl.link_views()
aviz.show(pl)
pd.DataFrame({f"{B} blades": {"solidity": PROPS[B].solidity, "blade area / disc": PROPS[B].solidity} for B in MANY}).round(3)

## 2. The installation — the same model as notebook 25

The air the blades meet (`air_propeller.installation_wake`: the pod and the pylon's potential flow, and behind them — the
pusher — their viscous wakes) and the drag the running propeller adds to the pod and the pylon (`installation_drag`): the
installed operating point at any rpm and airspeed (`model_point`), the rpm where the aircraft flies (`flying_rpm`) and the
point at full power (`full_power`: the reference shaft power, or the tip-Mach rpm, whichever comes first).

In [ ]:
R_MM = BLADE["diameter_mm"] / 2
INST_WAKE = {(lay, V): ap.installation_wake(pod_p(lay), R_MM, V, cd=PYLON_CD, nu=NU) for lay in LAYOUTS for V in SPEEDS}

def model_point(kind, B, rpm, V):
    wk = INST_WAKE.get((kind, V)) or ap.installation_wake(pod_p(kind), R_MM, V, cd=PYLON_CD, nu=NU)
    w, op = wake.effective_inflow(PROPS[B], SECTION, rpm, V, wk, RHO)
    dD = ap.installation_drag(pod_p(kind), R_MM, V, op.thrust, rho=RHO, nu=NU, cd_pylon=PYLON_CD, loading=(op.r, op.dT_dr))["total_N"]
    ok = op.power > 0 and op.thrust > 0
    return {"rpm": rpm, "V": V, "thrust": op.thrust, "net_thrust": op.thrust - dD, "power": op.power, "w_eff": w,
            "t": dD / op.thrust if op.thrust > 0 else np.nan, "eta": op.thrust * V / op.power if ok else np.nan,
            "eta_net": (op.thrust - dD) * V / op.power if ok and op.thrust > dD else np.nan}

def tip_rpm(B):
    return TIP_MACH_LIMIT * 340.0 / (math.pi * PROPS[B].diameter) * 60

def flying_rpm(kind, B, V):
    lo, hi, need = 300.0, tip_rpm(B), required_thrust(V)
    for _ in range(22):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if model_point(kind, B, mid, V)["net_thrust"] < need else (lo, mid)
    return 0.5 * (lo + hi)

def full_power(kind, B, V):
    hi = model_point(kind, B, tip_rpm(B), V)
    if hi["power"] <= P_REF_SHAFT:
        return hi
    lo_, hi_ = 500.0, tip_rpm(B)
    for _ in range(18):
        mid = 0.5 * (lo_ + hi_)
        lo_, hi_ = (mid, hi_) if model_point(kind, B, mid, V)["power"] <= P_REF_SHAFT else (lo_, mid)
    return model_point(kind, B, lo_, V)

V_FULL = (0.5, 10.0, 20.0, 30.0, 40.0, 50.0)
LEVEL = {(lay, B): [model_point(lay, B, flying_rpm(lay, B, V), V) for V in SPEEDS] for lay in LAYOUTS for B in MANY}
FULL = {(lay, B): [full_power(lay, B, V) for V in V_FULL] for lay in LAYOUTS for B in MANY}
if COMPARE:                                         # notebook 25's two and three blades, the same definitions
    for key, c in PJ["configs"].items():
        LEVEL[(c["layout"], c["blades"])] = c["level_flight"]
        FULL[(c["layout"], c["blades"])] = c["full_power"]
BLADE_SET = sorted({B for _, B in LEVEL})
print("blade counts compared:", BLADE_SET, f"| full power: {P_REF_SHAFT:.0f} W shaft, tip Mach {TIP_MACH_LIMIT}")

## 3. Six and twelve blades against two and three

Left: the **net thrust at full power** against airspeed (the launch, the climb, the top speed). Middle: the **net efficiency
in level flight** of the 3 kg aircraft (cruise and range). Right: the **blade-passing frequency** where it flies. Solid:
tractor, dashed: pusher.

In [ ]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(1, 3, figsize=(21, 5.0))
for (lay, B), pts in sorted(FULL.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    lv = LEVEL[(lay, B)]
    series = ((ax[0], [p["V"] for p in pts], [p["net_thrust"] for p in pts]),
              (ax[1], [p["V"] for p in lv], [p["eta_net"] for p in lv]),
              (ax[2], [p["V"] for p in lv], [p["rpm"] * B / 60 for p in lv]))
    for a, x, y in series:
        a.plot(x, y, color=BLADE_COLOR[B], **LAYOUT_STYLE[lay])
        if lay == "pusher":                                       # name the line where it ends
            a.annotate(f"{B}", (x[-1], y[-1]), xytext=(6, 0), textcoords="offset points", va="center", fontsize=10,
                       color=BLADE_COLOR[B], weight="bold")
ax[0].set(xlabel="airspeed [m/s]", ylabel="net thrust [N]", title=f"full power ({P_REF_SHAFT:.0f} W shaft, tip Mach {TIP_MACH_LIMIT})")
ax[1].set(xlabel="airspeed [m/s]", ylabel="net efficiency (T − ΔD) V / P", title="level flight, where the aircraft flies", ylim=(0, 0.9))
ax[2].set(xlabel="airspeed [m/s]", ylabel="Hz", title="blade-passing frequency in level flight", yscale="log")
handles = [Line2D([], [], color=BLADE_COLOR[B], lw=3, label=f"{B} blades") for B in BLADE_SET] + \
          [Line2D([], [], color="#555555", label=lay, **LAYOUT_STYLE[lay]) for lay in ("pusher", "tractor")]
for a in ax:
    a.grid(color="#e8e8e8", lw=0.6); a.margins(x=0.08)
fig.legend(handles=handles, loc="upper center", ncol=len(handles), frameon=False, fontsize=10, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.93))
rows = {}
for (lay, B) in sorted(FULL, key=lambda k: (k[1], k[0])):
    f, lv = FULL[(lay, B)], LEVEL[(lay, B)]
    l20 = min(lv, key=lambda p: abs(p["V"] - POINTS["cruise"][0]))
    f40 = min(f, key=lambda p: abs(p["V"] - 40.0))
    rows[(f"{B} blades", lay)] = {"static net thrust [N]": f[0]["net_thrust"], "net thrust at 40 m/s [N]": f40["net_thrust"],
                                  "rpm at full power, static": f[0]["rpm"], f"η net at {l20['V']:.0f} m/s": l20["eta_net"],
                                  f"rpm at {l20['V']:.0f} m/s": l20["rpm"], "blade passing [Hz]": l20["rpm"] * B / 60,
                                  "effective wake w": l20["w_eff"], "thrust deduction t": l20["t"],
                                  "source": "notebook 25" if B not in MANY else "this notebook"}
MANY_TAB = pd.DataFrame(rows).T
MANY_TAB

**Reading the blade count.** At the same diameter and pitch, more blades absorb the same power at a lower rpm: at full power
the two-blade propeller turns near 18 000 rpm, the twelve-blade one near 11 000. That buys **static thrust** — about 47.7 N
with two blades, 50 with three, 53 with six and twelve (the launch and the climb). **At speed** the order changes: at 40 m/s
three blades give the most (about 27 N), six about 25, two about 23 — the two-blade propeller is held back near its tip-speed
limit — and twelve fall to about 20, so lightly loaded that the profile drag of all that blade area dominates. **In level
flight** efficiency falls with every blade added — at 20 m/s about 0.70 with two, 0.68 with three, 0.61 with six, 0.49 with
twelve (each blade at a lower lift for the same total thrust, all the blade area still dragging) — and the **tone** moves up:
the blade-passing frequency rises from about 300 Hz to 1.5 kHz, where the ear weights it most. The layouts keep their order:
the pusher a little ahead of the tractor at every blade count. So for a sprint three blades are the sweet spot, for range two;
six and twelve buy launch thrust and a lower tip speed, not speed or range. MERLIN's race (notebook 27) has them all.

## 4. The exotic-propeller maps for the flight models

Six- and twelve-blade propellers of the same design space as notebook 25's maps — 9–12 inch, pitch 0.6–1.2 diameters — over
airspeed × rpm, with their installation on MERLIN, written to `designs/data/exotic_propeller_maps.json`
(`designs/propulsor_maps.py`, about a minute). The committed file is loaded unless `AIR_PROP_MAPS=build`.

In [ ]:
import propulsor_maps as pmaps
if os.environ.get("AIR_PROP_MAPS") == "build" or not pmaps.LIBRARIES["exotic"].exists():
    print("written", pmaps.save(pmaps.build_library("exotic", progress=True)))
XMAPS = pmaps.load(pmaps.LIBRARIES["exotic"])
rows = {}
for e in XMAPS["entries"]:
    if "error" in e:
        continue
    for lay in LAYOUTS:
        u = pmaps.unit(e, 1900.0, layout=lay)
        rows[(e["label"], lay)] = {"static thrust at 1900 W [N]": u.full(0.0)[0], "at 30 m/s [N]": u.full(30.0)[0], "at 60 m/s [N]": u.full(60.0)[0]}
print(f"{len(XMAPS['entries'])} exotic propeller maps in {pmaps.LIBRARIES['exotic']}")
pd.DataFrame(rows).T.sort_values("at 60 m/s [N]", ascending=False).head(12).round(2)